# Importing all functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# creating incremental flag

In [0]:
dbutils.widgets.text("incremental_flag",'0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')

# connection to ADLS

In [0]:
spark.conf.set("fs.azure.account.key.saleessa.dfs.core.windows.net","access_key")

# defining paths

In [0]:
silver_path = "abfss://silver@saleessa.dfs.core.windows.net/"
gold_path = "abfss://gold@saleessa.dfs.core.windows.net/"

# selecting product data

In [0]:
df_src = spark.sql(
    ''' select distinct(Sale_Date) as sale_date,Year,Month,Quarter from 
        parquet.`abfss://silver@saleessa.dfs.core.windows.net/`
    
    '''
)

# creating sink dataset

In [0]:
from delta.tables import DeltaTable

path =  "abfss://gold@saleessa.dfs.core.windows.net/dim_date"

if DeltaTable.isDeltaTable(spark,path):
    df_sink = spark.read.format('delta').load(path).select("date_key","sale_date","Year","Month","Quarter")
else:
    df_sink = spark.createDataFrame([], "date_key int,sale_date date,Year int,Month int,Quarter int")
 


In [0]:
df = df_src.join(df_sink,on='sale_date',how='left').select(df_src.sale_date,df_src.Year,df_src.Month,df_src.Quarter,df_sink.date_key)


sale_date,Year,Month,Quarter,date_key
2024-01-24,2024,1,1,null
2024-01-30,2024,1,1,null
2024-01-17,2024,1,1,null
2024-01-01,2024,1,1,null
2024-01-11,2024,1,1,null
2024-01-05,2024,1,1,null
2024-01-02,2024,1,1,null
2024-01-25,2024,1,1,null
2024-01-23,2024,1,1,null
2024-01-09,2024,1,1,null


# old and new records

In [0]:
df_old = df.filter(df.date_key.isNotNull())
df_new = df.filter(df.date_key.isNull())

In [0]:
if incremental_flag=='0':
    max_value = 1
else:
    max_value = df_old.select(max(df_old.date_key)).collect()[0][0]

max_value

1

# adding serogate keys to new data

In [0]:
df_new = df_new.withColumn('date_key',max_value + monotonically_increasing_id())

In [0]:
union_df = df_old.union(df_new)

In [0]:
path = "abfss://gold@saleessa.dfs.core.windows.net/dim_date"


if DeltaTable.isDeltaTable(spark,path):
    deltatable = DeltaTable.forPath(spark,path)
    deltatable.alias('target').merge(union_df.alias('source'),'target.date_key = source.date_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('file upserted')

else:
    union_df.write.format('delta').mode('overwrite').option('mergeSchema','true').save(path)
    print('file created')

file created
